# Module 6 project: Grover's search with noise

**Quantum Computing Intermediate · Module 6 · about 180 minutes** · notebook version 2026-10-05

**The question of this experiment.** Without noise, 2 Grover iterations find one marked string among 8 with probability 0.945. Each iteration adds two 3-qubit controlled-Z gates, 12 CNOTs in all, and each CNOT adds a little error. **On a noisy computer, is 2 still the best number of iterations, and how noisy can the computer be before 1 iteration wins?**

In this project you run an experiment on a **noisy simulator**: a simulator that adds the errors of a real quantum computer, described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain each result is, and say what the experiment cannot tell you. These are the skills of course outcome 5.

**The plan.**

1. Write three statistics functions (Step 1) and meet the course noise model (Step 2).
2. Write `grover_circuit(marked, iterations)`, your Module 2 search as one function (Step 3).
3. Run it once on the noisy simulator and compare the result with the noise model's expected value (Step 4).
4. Measure the success probability for 0 to 5 iterations, with uncertainties (Step 5), and test whether 2 iterations beat 1 (Step 6).
5. Find how noisy the computer can be before 1 iteration wins (Step 7).
6. Run your own computer's noise model and get your verification value (Step 8), then look at 2 qubits (Step 9) and at the limits.

**How it is graded.** Your project quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell, three results from this notebook, and four questions about uncertainty, interpretation and limits. Your project quiz gives you three numbers: MARKED (the string to search for, 0 to 7), and P2 and READOUT for your computer.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import (QuantumCircuit, Statevector, AerSimulator, NoiseModel, depolarizing_error, ReadoutError,
                  simulate_density_matrix)

print("Ready. qsim", qsim.__version__, "and NumPy", np.__version__)

## Step 1: how certain is a result from shots?

A circuit that gives a result with probability p, run for S shots, gives that result k times. The number k is random: it follows the **binomial distribution**. The fraction f = k / S has

- mean p, and
- standard deviation **σ = √(p (1 − p) / S)**. This spread is called **shot noise**.

For example, with S = 4,000 shots and p = 0.687, σ = 0.0073: two runs of the same circuit typically differ by about this much. Four times as many shots halve σ.

**The course rule: 3σ.** A measured fraction f agrees with an expected value p when |f − p| ≤ 3σ. For a noisy simulator, p must be the **noise model's** expected value (computed exactly, without shots, in Step 2), not the ideal value from a circuit without noise. A correct run lands outside 3σ only about 3 times in 1,000 (0.27%).

**Comparing two results.** Two fractions f₁ and f₂, each from S independent shots, differ by f₂ − f₁, with standard deviation

**σ_diff = √( f₁(1 − f₁)/S + f₂(1 − f₂)/S )**.

Uncertainties of independent results add **in quadrature** (their squares add), so a difference is less certain than either result alone. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but this experiment cannot show it.

**Your task:** write the three functions below. They are used in every later step, and the check cell tests them.

- `shot_sigma(p, shots)`: σ = √(p (1 − p) / shots), the standard deviation of a fraction.
- `within_3_sigma(k, shots, p)`: `True` if the fraction k / shots is within 3 · shot_sigma(p, shots) of p (in either direction), otherwise `False`.
- `difference_sigma(f1, f2, shots)`: σ_diff for two fractions from `shots` shots each.

In [ ]:
def shot_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete shot_sigma() first.")


def within_3_sigma(k, shots, p):
    """True if k successes out of `shots` agree with the probability p by the course's 3-sigma rule."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete within_3_sigma() first.")


def difference_sigma(f1, f2, shots):
    """The standard deviation of f2 - f1, for two fractions from `shots` independent shots each."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete difference_sigma() first.")

In [ ]:
print("shot_sigma(0.687, 4000) =", round(shot_sigma(0.687, 4000), 5), "  (expected 0.00733)")
print("within_3_sigma(2748, 4000, 0.687) =", within_3_sigma(2748, 4000, 0.687), "  (expected True: exactly 0.687)")
print("within_3_sigma(2655, 4000, 0.687) =", within_3_sigma(2655, 4000, 0.687), "  (expected False: 3.2 sigma too low)")
print("difference_sigma(0.687, 0.634, 4000) =", round(difference_sigma(0.687, 0.634, 4000), 5), "  (expected 0.01057)")

## Step 2: the course noise model

A **noise model** says what can go wrong in each operation. The course model has three parts, each with a number:

- after every **two-qubit gate** (`cx`, `cz`, `cp`, `rzz`, `swap`), a **depolarizing error** with parameter P2: with probability P2 the two qubits are replaced by a completely random state. For a density matrix, ρ → (1 − P2) ρ + P2 · I/4 on those two qubits. (An average gate error, the number IBM reports, of 0.75 · P2.)
- after every **one-qubit gate**, the same on one qubit with parameter P2 / 10: one-qubit gates are about ten times better.
- at **readout**: a qubit in |0⟩ is recorded as 1 with probability READOUT, and a qubit in |1⟩ is recorded as 0 with probability 2 · READOUT. The second is larger because an excited qubit can decay to |0⟩ during the measurement.

The course values are **P2 = 0.01** and **READOUT = 0.02**. They are several times worse than IBM's best current computers (the model of `ibm_pittsburgh` used in Module 4 has a median two-qubit gate error of 0.0015 and a median readout error of 0.0045), so that the effects of noise show clearly in a few thousand shots. In Canvas, your project quiz gives you your own P2 (0.005 to 0.030) and READOUT (0.005 to 0.030): a model of "your" computer.

The noise model applies errors by **gate name**. A gate on three qubits, such as `ccx` or `mcx`, would get no error at all, so every circuit in this project uses only one- and two-qubit gates.

The prepared cell builds the noise model with the same functions as Qiskit Aer (`qiskit_aer.noise`), and defines `expected_probabilities(qc, p2, readout)`. It gives the noise model's **exact** probability of every recorded result, with no shots and no randomness: it follows the **density matrix**, the mixed state that the noisy gates produce, then applies the readout errors to the recorded bits. A run with shots, `AerSimulator(noise_model=...).run(qc, shots=4000)`, samples from these same probabilities and adds shot noise. The 3σ rule compares the two.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]


def course_noise_model(p2=0.01, readout=0.02):
    """Prepared: depolarizing error p2 after two-qubit gates and p2/10 after one-qubit gates; readout errors."""
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    nm.add_all_qubit_readout_error(ReadoutError([[1 - readout, readout], [2 * readout, 1 - 2 * readout]]))
    return nm


def expected_probabilities(qc, p2=0.01, readout=0.02):
    """Prepared: the noise model's exact probability of each recorded result, as a dict {"bits": probability}.

    qc must end with its measurements (classical bit j receives one qubit). No shots are taken.
    """
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    rho = simulate_density_matrix(body, course_noise_model(p2, readout))   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])      # index bit j = classical bit j
    confusion = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])   # rows: true bit; columns: recorded bit
    for j in range(m):                                                     # each recorded bit can be flipped
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, confusion).reshape(-1)
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


print(course_noise_model())

## Step 3: your Grover circuit

**Your task:** write `grover_circuit(marked, iterations)`, as in Module 2: H on every qubit; then `iterations` rounds of

- the **oracle**: X on every qubit whose bit of `marked` is 0, the multi-controlled Z `mcz`, the same X gates;
- the **diffuser**: H, X, `mcz`, X, H on every qubit;

then measure qubit q into classical bit q. Qubit q is the bit `marked[n - 1 - q]` (qubit 0 is the rightmost bit), and `n = len(marked)` is 2 or 3.

The prepared `mcz` is `cz` for 2 qubits. For 3 qubits it is **not** the `h`, `mcx`, `h` of Module 2: a 3-qubit gate would get no error from the noise model, so `mcz` is written out with 6 CNOTs and 7 T or T† gates, as a real computer would run it. Use it once in each oracle and once in each diffuser.

The cell after it checks the ideal success probability for 101 with 2 iterations (no noise): 0.9453.

In [ ]:
def mcz(qc, qubits):
    """Prepared: a controlled Z on 2 or 3 qubits, written with one- and two-qubit gates only."""
    qubits = list(qubits)
    if len(qubits) == 2:
        qc.cz(qubits[0], qubits[1])
        return
    a, b, c = qubits                       # the textbook CCZ: 6 CNOTs, T and T-dagger gates
    qc.cx(b, c); qc.tdg(c); qc.cx(a, c); qc.t(c); qc.cx(b, c); qc.tdg(c); qc.cx(a, c)
    qc.t(b); qc.t(c); qc.cx(a, b); qc.t(a); qc.tdg(b); qc.cx(a, b)


def grover_circuit(marked, iterations):
    """Grover's search for one marked bit string, with measurements (your Module 2 code as one function)."""
    n = len(marked)
    qc = QuantumCircuit(n, n)
    # YOUR CODE HERE: H on every qubit; `iterations` rounds of the oracle and the diffuser; measure qubit q into bit q
    raise NotImplementedError("Complete grover_circuit() first.")
    return qc

In [ ]:
qc = grover_circuit("101", 2)
ideal = expected_probabilities(qc, p2=0, readout=0)
print("ideal P(101) with 2 iterations:", round(ideal.get("101", 0), 4), "  (expected 0.9453)")
print("gates:", qc.count_ops())

**What to notice.** With 2 iterations the circuit has 4 `mcz` gates, so 24 `cx` gates, plus 16 `t` and 12 `tdg` gates and the H and X gates. `expected_probabilities(qc, p2=0, readout=0)` gives the ideal probabilities: a noise model with zero errors.

## Step 4: one run on the noisy simulator

Now use the course noise model (P2 = 0.01, READOUT = 0.02). The cell

1. computes the noise model's **expected** probability of 101, `p_expected`, exactly;
2. runs **4,000 shots** on `AerSimulator(noise_model=course_noise_model())`;
3. compares the two with your `within_3_sigma`, and plots the expected and measured fraction of every result.

`RUN_SEED` fixes the simulator's random numbers so that you can repeat a run. Change it and run the cell again: the measured count changes, the expected value does not. The project quiz asks for your measured fraction of 101 (any run you made with 4,000 shots) and for σ.

In [ ]:
RUN_SEED = 1                     # any whole number; change it to see another run
noise = course_noise_model(0.01, 0.02)
sim = AerSimulator(noise_model=noise)

qc = grover_circuit("101", 2)
p_expected = expected_probabilities(qc, 0.01, 0.02)["101"]
counts = sim.run(qc, shots=4000, seed_simulator=RUN_SEED).result().get_counts()
k = counts.get("101", 0)
sigma = shot_sigma(p_expected, 4000)
print(f"ideal (no noise):               P(101) = {ideal['101']:.4f}")
print(f"noise model, expected:          P(101) = {p_expected:.4f}")
print(f"noise model, 4000 shots:        {k} found 101, a fraction {k / 4000:.4f}")
print(f"sigma = {sigma:.4f}, 3 sigma = {3 * sigma:.4f}; the measured fraction is {(k / 4000 - p_expected) / sigma:+.2f} sigma from expected")
print("within 3 sigma:", within_3_sigma(k, 4000, p_expected))

expected = expected_probabilities(qc, 0.01, 0.02)
labels = [format(i, "03b") for i in range(8)]
x = np.arange(8)
plt.figure(figsize=(7, 3.2))
plt.bar(x - 0.2, [expected.get(s, 0) for s in labels], 0.4, label="noise model, expected")
plt.bar(x + 0.2, [counts.get(s, 0) / 4000 for s in labels], 0.4, label="4000 shots")
plt.xticks(x, labels); plt.ylabel("fraction of shots"); plt.legend(); plt.tight_layout(); plt.show()

**What to notice.** The noise takes P(101) from 0.945 down to 0.687. The lost probability is spread over the seven wrong strings: depolarizing errors push the state toward a completely random one, where every string has probability 1/8. Your measured fraction should lie within about 0.022 (3σ) of 0.687; compare it with the ideal 0.945 too, which is 35σ away: a noisy run cannot be judged against the ideal value.

## Step 5: success against the number of iterations

The cell computes, for 0 to 5 iterations, the ideal probability of 101, the noise model's expected probability, and a measured fraction from 4,000 shots with its σ. The plot shows the measured fractions as points with **error bars of ±3σ**. It takes a few seconds.

In [ ]:
iterations = list(range(6))
ideal_p, noisy_p, measured, sig = [], [], [], []
for it in iterations:
    qc = grover_circuit("101", it)
    ideal_p.append(expected_probabilities(qc, 0, 0)["101"])
    noisy_p.append(expected_probabilities(qc, 0.01, 0.02)["101"])
    kk = sim.run(qc, shots=4000, seed_simulator=RUN_SEED + 10 + it).result().get_counts().get("101", 0)
    measured.append(kk / 4000)
    sig.append(shot_sigma(noisy_p[-1], 4000))

print("iterations   ideal    expected (noise)   measured    sigma    cx gates")
for it in iterations:
    print(f"    {it}       {ideal_p[it]:.4f}       {noisy_p[it]:.4f}         {measured[it]:.4f}    {sig[it]:.4f}      {12 * it}")

plt.figure(figsize=(7, 3.5))
plt.plot(iterations, ideal_p, "k--o", label="ideal")
plt.plot(iterations, noisy_p, "C0-s", label="noise model, expected")
plt.errorbar(iterations, measured, yerr=[3 * s for s in sig], fmt="C3o", capsize=4, label="4000 shots, ±3σ")
plt.axhline(1 / 8, color="gray", lw=1, ls=":", label="random guess (1/8)")
plt.xlabel("iterations"); plt.ylabel("P(101)"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

**What to notice.**

- Noise lowers the peak (0.945 to 0.687 at 2 iterations) and every iteration costs more: each adds 12 `cx` gates.
- Where the ideal probability is **below** 1/8 (4 iterations: 0.012), the noise **raises** it (to 0.070). Noise does not simply subtract: it mixes the state toward the random one, which pulls every probability toward 1/8.
- 2 iterations still give the highest expected value under the course noise model. Is the measured difference from 1 iteration significant? That is Step 6.

## Step 6: is 2 iterations better than 1?

Use `difference_sigma` to test the difference between the measured fractions of 1 and 2 iterations from Step 5. The difference is **significant** if it is larger than 3σ_diff.

In [ ]:
f1, f2 = measured[1], measured[2]
d = f2 - f1
sd = difference_sigma(f1, f2, 4000)
print(f"1 iteration: {f1:.4f}   2 iterations: {f2:.4f}   difference {d:+.4f}")
print(f"sigma_diff = {sd:.4f}; the difference is {d / sd:.1f} sigma_diff")
print("significant (more than 3 sigma_diff):", abs(d) > 3 * sd)
print(f"expected difference from the noise model: {noisy_p[2] - noisy_p[1]:+.4f}")

**What to notice.** With the course noise model the expected difference is 0.053, about 5σ_diff for 4,000 shots each, so your measured difference will almost always be significant: under this noise, 2 iterations are better. The test uses only the two measured fractions; it would work the same way on a real computer, where no one knows the expected values.

## Step 7: how noisy can the computer be?

As P2 grows, the 24 `cx` gates of 2 iterations lose more than the 12 of 1 iteration. The cell scans P2 from 0 to 0.030 in steps of 0.001 (READOUT stays 0.02) and computes the noise model's expected success for 1 and 2 iterations; no shots are needed. It prints the **crossover**: the smallest P2 in the scan at which 1 iteration is better. The project quiz asks for it.

Then it asks how many shots would show the difference at P2 = 0.015. For a difference d between two fractions near p₁ and p₂, the condition d > 3σ_diff with S shots each gives **S > 9 (p₁(1 − p₁) + p₂(1 − p₂)) / d²**.

In [ ]:
p2_values = np.round(np.arange(0, 0.0305, 0.001), 3)
one = [expected_probabilities(grover_circuit("101", 1), p, 0.02)["101"] for p in p2_values]
two = [expected_probabilities(grover_circuit("101", 2), p, 0.02)["101"] for p in p2_values]
crossover = next(p for p, a, b in zip(p2_values, one, two) if a > b)
print(f"crossover: from P2 = {crossover:.3f}, 1 iteration gives a higher expected success than 2")

i = list(p2_values).index(0.015)
a, b = one[i], two[i]
shots_needed = 9 * (a * (1 - a) + b * (1 - b)) / (b - a) ** 2
print(f"at P2 = 0.015: 1 iteration {a:.4f}, 2 iterations {b:.4f}, difference {b - a:+.4f}")
print(f"  with 4000 shots each, sigma_diff = {difference_sigma(a, b, 4000):.4f}; shots needed for 3 sigma_diff: about {shots_needed:,.0f} each")

plt.figure(figsize=(7, 3.3))
plt.plot(p2_values, one, label="1 iteration"); plt.plot(p2_values, two, label="2 iterations")
plt.axvline(crossover, color="gray", ls=":", lw=1)
plt.xlabel("P2 (two-qubit depolarizing error)"); plt.ylabel("expected P(101)"); plt.legend(); plt.tight_layout(); plt.show()

**What to notice.** The two curves cross near P2 = 0.018 (an average two-qubit gate error of about 0.013): on a computer noisier than that, the shorter circuit wins although it is worse in theory. Near the crossover the two are almost equal, and telling them apart takes many shots: about 14,000 each at P2 = 0.015, where the difference is 0.017. Close to a crossover, a "which is better" question needs far more data than one far from it.

## Step 8: your computer, and your verification value

Open your **project quiz for this experiment** in Canvas. Question 1 shows your MARKED (0 to 7: the 3-bit string to search for, for example 5 is 101), your P2 and your READOUT. Type them below and run the cells.

The first cell runs 1 and 2 iterations on **your** noise model, 4,000 shots each, and keeps the 2-iteration counts as `my_counts`. The check cell then tests your three statistics functions and `grover_circuit()`, and checks that `my_counts` agree with your noise model by the 3σ rule. Only if everything passes does it print your **verification value**: 10,000 times the noise model's exact probability that 2 iterations find MARKED on your computer, rounded to a whole number.

In [ ]:
MARKED = -1      # your number from Canvas, for example 5 (the string 101)
P2 = 0.0         # your number from Canvas, for example 0.017
READOUT = 0.0    # your number from Canvas, for example 0.015
my_counts = {}   # filled by the next cell

In [ ]:
my_marked = format(int(MARKED), "03b")
my_sim = AerSimulator(noise_model=course_noise_model(P2, READOUT))
my_runs = {}
for it in (1, 2):
    qc = grover_circuit(my_marked, it)
    p = expected_probabilities(qc, P2, READOUT)[my_marked]
    cts = my_sim.run(qc, shots=4000, seed_simulator=RUN_SEED + 100 + it).result().get_counts()
    my_runs[it] = cts
    kk = cts.get(my_marked, 0)
    print(f"{it} iteration(s): expected {p:.4f}, measured {kk / 4000:.4f} ± {shot_sigma(p, 4000):.4f} (1 sigma)")
my_counts = my_runs[2]
f1, f2 = my_runs[1].get(my_marked, 0) / 4000, my_runs[2].get(my_marked, 0) / 4000
print(f"difference (2 minus 1): {f2 - f1:+.4f} = {(f2 - f1) / difference_sigma(f1, f2, 4000):+.1f} sigma_diff")

In [ ]:
# The course's check code. You do not need to read it; it is here so that you can.
import math

import numpy as np

try:                                    # real Qiskit, if it is installed
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
except ImportError:                     # the browser: the course simulator
    from qsim import QuantumCircuit, Statevector

ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
SHOTS = 4000

# ---------------------------------------------------------------- an exact noisy simulator (NumPy only)
_S2 = 1 / math.sqrt(2)
_FIXED = {
    "h": np.array([[_S2, _S2], [_S2, -_S2]], dtype=complex),
    "x": np.array([[0, 1], [1, 0]], dtype=complex),
    "t": np.diag([1, np.exp(1j * math.pi / 4)]),
    "tdg": np.diag([1, np.exp(-1j * math.pi / 4)]),
    "s": np.diag([1, 1j]),
    "sdg": np.diag([1, -1j]),
}


def _matrix(name, params):
    """The gate's matrix in Qiskit's convention (for two-qubit gates, the first qubit is the right-hand factor)."""
    if name in _FIXED:
        return _FIXED[name]
    if name == "rx":
        c, s = math.cos(params[0] / 2), math.sin(params[0] / 2)
        return np.array([[c, -1j * s], [-1j * s, c]])
    if name == "rz":
        return np.diag([np.exp(-0.5j * params[0]), np.exp(0.5j * params[0])])
    if name == "p":
        return np.diag([1, np.exp(1j * params[0])])
    if name == "cx":                     # control = first qubit (the low bit of the 4 x 4 index)
        return np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]], dtype=complex)
    if name == "cz":
        return np.diag([1, 1, 1, -1]).astype(complex)
    if name == "cp":
        return np.diag([1, 1, 1, np.exp(1j * params[0])])
    if name == "rzz":
        a = np.exp(-0.5j * params[0])
        b = np.exp(0.5j * params[0])
        return np.diag([a, b, b, a])
    if name == "swap":
        return np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)
    raise ValueError(f"unknown gate {name}")


def _apply(rho, u, qubits, n):
    """rho -> U rho U^dagger, with U acting on the given qubits (qubits[0] is the low bit of U's index)."""
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    # axis of qubit q in the row index is n - 1 - q; in the column index 2n - 1 - q
    ut = u.reshape([2] * (2 * k))           # axes: out bits (high to low), in bits (high to low)
    row_axes = [n - 1 - q for q in reversed(qubits)]
    col_axes = [2 * n - 1 - q for q in reversed(qubits)]
    t = np.tensordot(ut, t, axes=(list(range(k, 2 * k)), row_axes))
    t = np.moveaxis(t, list(range(k)), row_axes)
    t = np.tensordot(t, ut.conj(), axes=(col_axes, list(range(k, 2 * k))))
    t = np.moveaxis(t, list(range(2 * n - k, 2 * n)), col_axes)
    return t.reshape(2 ** n, 2 ** n)


def _depolarize(rho, lam, qubits, n):
    """rho -> (1 - lam) rho + lam (I / 2^k on the given qubits) x (rho traced over them)."""
    if lam == 0:
        return rho
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    row_axes = [n - 1 - q for q in qubits]
    col_axes = [2 * n - 1 - q for q in qubits]
    letters = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    sub = list(letters[:2 * n])
    for ra, ca in zip(row_axes, col_axes):
        sub[ca] = sub[ra]
    keep = [i for i in range(2 * n) if i not in row_axes + col_axes]
    red = np.einsum("".join(sub) + "->" + "".join(sub[i] for i in keep), t)
    eye = np.eye(2 ** k).reshape([2] * (2 * k))      # identity: row bit i pairs with column bit i
    full = np.multiply.outer(red, eye)
    full = np.moveaxis(full, list(range(len(keep), 2 * n)), row_axes[::-1] + col_axes[::-1])
    return (1 - lam) * rho + lam * full.reshape(2 ** n, 2 ** n) / 2 ** k


def exact_probabilities(ops, n, measured, p2, readout):
    """Exact probabilities of the recorded results under the course noise model.

    ops: list of (name, qubits, params) applied to |0...0>; measured: the qubits measured at the end, measured[j] into
    classical bit j. Returns an array over the 2^len(measured) recorded results (bit j of the index is classical bit j).
    """
    rho = np.zeros((2 ** n, 2 ** n), dtype=complex)
    rho[0, 0] = 1
    for name, qubits, params in ops:
        rho = _apply(rho, _matrix(name, params), list(qubits), n)
        if name in TWO_QUBIT_GATES:
            rho = _depolarize(rho, p2, list(qubits), n)
        elif name in ONE_QUBIT_GATES:
            rho = _depolarize(rho, p2 / 10, list(qubits), n)
    probs = np.real(np.diag(rho)).clip(min=0)
    probs = probs / probs.sum()
    m = len(measured)
    idx = np.arange(2 ** n)
    code = np.zeros(2 ** n, dtype=int)
    for j, q in enumerate(measured):
        code |= ((idx >> q) & 1) << j
    dist = np.zeros(2 ** m)
    np.add.at(dist, code, probs)
    conf = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])
    t = dist.reshape([2] * m)
    for j in range(m):
        ax = m - 1 - j
        t = np.moveaxis(np.tensordot(t, conf, axes=([ax], [0])), -1, ax)
    dist = t.reshape(-1).clip(min=0)
    return dist / dist.sum()


def build_circuit(ops, n, measured):
    """The ops as a QuantumCircuit (Qiskit or qsim), with the measurements at the end."""
    qc = QuantumCircuit(n, len(measured)) if measured else QuantumCircuit(n)
    for name, qubits, params in ops:
        getattr(qc, name)(*params, *qubits)
    for j, q in enumerate(measured):
        qc.measure(q, j)
    return qc


# ---------------------------------------------------------------- helpers for the checks
def _op(inst):
    return getattr(inst, "operation", inst)


def _name(inst):
    return getattr(_op(inst), "name", "")


def _qubit_indices(qc, inst):
    qs = getattr(inst, "qubits", ())
    try:
        return [qc.find_bit(q).index for q in qs]
    except Exception:  # noqa: BLE001
        return [int(q) for q in qs]


def _clbit_indices(qc, inst):
    cs = getattr(inst, "clbits", ())
    try:
        return [qc.find_bit(c).index for c in cs]
    except Exception:  # noqa: BLE001
        return [int(c) for c in cs]


def measurement_map(qc):
    """{qubit: classical bit} for the measurements in a circuit."""
    return {_qubit_indices(qc, i)[0]: _clbit_indices(qc, i)[0] for i in qc.data if _name(i) == "measure"}


def big_gates(qc):
    """Names of gates on 3 or more qubits (the course noise model puts no error on them)."""
    return sorted({_name(i) for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) >= 3})


def two_qubit_count(qc):
    """The number of two-qubit gates in a circuit (barriers not counted)."""
    return sum(1 for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) == 2)


def ops_two_qubit_count(ops):
    return sum(1 for name, qubits, params in ops if len(qubits) == 2)


def ideal_probabilities(qc):
    """Probabilities of all qubits' values (index bit q = qubit q) for a circuit without its final measurements."""
    c = qc.copy()
    c.remove_final_measurements()
    return np.abs(np.asarray(Statevector(c).data, dtype=complex)) ** 2


def counts_total(counts):
    try:
        return int(sum(int(v) for v in counts.values()))
    except Exception:  # noqa: BLE001
        return -1


# ---------------------------------------------------------------- the statistics functions
def _close(a, b, tol=1e-9):
    try:
        return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
    except (TypeError, ValueError):
        return False


def test_statistics(shot_sigma, within_3_sigma, difference_sigma):
    """Returns (passed, messages)."""
    msgs = []
    cases = [(0.5, 100), (0.687, 4000), (0.1, 1000), (0.0, 50), (1.0, 50), (0.9206, 4000)]
    for p, n in cases:
        want = math.sqrt(p * (1 - p) / n)
        try:
            got = shot_sigma(p, n)
        except NotImplementedError:
            return False, ["shot_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"shot_sigma({p}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            hint = ""
            if _close(got, math.sqrt(p * (1 - p) * n)):
                hint = " That is the uncertainty of the count k; divide by shots for the fraction: sqrt(p * (1 - p) / shots)."
            elif _close(got, p * (1 - p) / n):
                hint = " Take the square root: sqrt(p * (1 - p) / shots)."
            elif _close(got, math.sqrt(p / n)) or _close(got, math.sqrt(1 / n)):
                hint = " Use p * (1 - p), the binomial variance of one shot."
            return False, [f"shot_sigma({p}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("shot_sigma(): passed (6 cases).")

    w_cases = [(2748, 4000, 0.687, True), (2655, 4000, 0.687, False), (2841, 4000, 0.687, False),
               (2700, 4000, 0.687, True), (2821, 4000, 0.687, True), (55, 100, 0.5, True), (66, 100, 0.5, False),
               (34, 100, 0.5, False), (3700, 4000, 0.9206, True), (3620, 4000, 0.9206, False)]
    for k, n, p, want in w_cases:
        try:
            got = within_3_sigma(k, n, p)
        except NotImplementedError:
            return False, msgs + ["within_3_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) raised {type(e).__name__}: {e}"]
        if bool(got) != want or not isinstance(got, (bool, np.bool_)):
            d = abs(k / n - p) / max(math.sqrt(p * (1 - p) / n), 1e-300)
            hint = ""
            if not isinstance(got, (bool, np.bool_)):
                hint = " Return True or False."
            elif k > n * p and not want:
                hint = " Check both directions: a result can be too high as well as too low (use abs())."
            elif 2 < d <= 3:
                hint = " The course rule allows up to 3 standard deviations."
            else:
                hint = " Compare the fraction k / shots with p, using shot_sigma(p, shots)."
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) returned {got}; expected {want} "
                                  f"(the result is {d:.2f} standard deviations from p).{hint}"]
    msgs.append("within_3_sigma(): passed (10 cases).")

    d_cases = [(0.687, 0.634, 4000), (0.5, 0.5, 100), (0.9, 0.1, 1000), (0.2, 0.25, 4000)]
    for f1, f2, n in d_cases:
        want = math.sqrt(f1 * (1 - f1) / n + f2 * (1 - f2) / n)
        try:
            got = difference_sigma(f1, f2, n)
        except NotImplementedError:
            return False, msgs + ["difference_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            s1, s2 = math.sqrt(f1 * (1 - f1) / n), math.sqrt(f2 * (1 - f2) / n)
            hint = ""
            if _close(got, s1 + s2):
                hint = " Uncertainties add in quadrature: sqrt(sigma1**2 + sigma2**2), not sigma1 + sigma2."
            elif _close(got, abs(s1 - s2)):
                hint = " The uncertainty of a difference is larger than either one: sqrt(sigma1**2 + sigma2**2)."
            elif _close(got, want ** 2):
                hint = " Take the square root."
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("difference_sigma(): passed (4 cases).")
    return True, msgs


def personal_noise_ok(P2, READOUT):
    """Checks the personal noise numbers from Canvas; returns (ok, message, p2, readout)."""
    try:
        p2, ro = float(P2), float(READOUT)
    except (TypeError, ValueError):
        return False, "P2 and READOUT must be the numbers shown in your Canvas project quiz.", None, None
    if not (0.005 - 1e-12 <= p2 <= 0.030 + 1e-12) or abs(round(p2, 3) - p2) > 1e-9:
        return False, "P2 is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    if not (0.005 - 1e-12 <= ro <= 0.030 + 1e-12) or abs(round(ro, 3) - ro) > 1e-9:
        return False, "READOUT is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    return True, "", round(p2, 3), round(ro, 3)


GROVER_CASES = [("11", 1), ("01", 1), ("10", 2), ("101", 2), ("000", 1), ("110", 3), ("011", 0), ("111", 2)]


def reference_mcz_ops(qubits):
    """The prepared mcz as a list of (name, qubits, params)."""
    if len(qubits) == 2:
        return [("cz", [qubits[0], qubits[1]], [])]
    a, b, c = qubits
    return [("cx", [b, c], []), ("tdg", [c], []), ("cx", [a, c], []), ("t", [c], []), ("cx", [b, c], []),
            ("tdg", [c], []), ("cx", [a, c], []), ("t", [b], []), ("t", [c], []), ("cx", [a, b], []),
            ("t", [a], []), ("tdg", [b], []), ("cx", [a, b], [])]


def reference_grover_ops(marked, iterations):
    n = len(marked)
    qs = list(range(n))
    ops = [("h", [q], []) for q in qs]
    zeros = [q for q in qs if marked[n - 1 - q] == "0"]
    for _ in range(iterations):
        ops += [("x", [q], []) for q in zeros]
        ops += reference_mcz_ops(qs)
        ops += [("x", [q], []) for q in zeros]
        ops += [("h", [q], []) for q in qs] + [("x", [q], []) for q in qs]
        ops += reference_mcz_ops(qs)
        ops += [("x", [q], []) for q in qs] + [("h", [q], []) for q in qs]
    return ops


def reference_grover(marked, iterations):
    n = len(marked)
    return build_circuit(reference_grover_ops(marked, iterations), n, list(range(n)))


def expected_success(marked, iterations, p2, readout):
    """Exact probability that the reference circuit records the marked string under the course noise model."""
    n = len(marked)
    dist = exact_probabilities(reference_grover_ops(marked, iterations), n, list(range(n)), p2, readout)
    return float(dist[int(marked, 2)])


def _test_grover(grover_circuit):
    for marked, it in GROVER_CASES:
        n = len(marked)
        try:
            qc = grover_circuit(marked, it)
        except NotImplementedError:
            return False, ["grover_circuit() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"grover_circuit('{marked}', {it}) raised {type(e).__name__}: {e}"]
        if not hasattr(qc, "data") or getattr(qc, "num_qubits", None) != n:
            return False, [f"grover_circuit('{marked}', {it}) should return a QuantumCircuit on {n} qubits."]
        big = big_gates(qc)
        if big:
            return False, [f"grover_circuit('{marked}', {it}) uses {', '.join(big)} on 3 qubits. Use the prepared mcz(qc, qubits): "
                           "it is written with cx, t and tdg, and the course noise model puts errors only on one- and "
                           "two-qubit gates, so a 3-qubit gate would run without any noise."]
        mm = measurement_map(qc)
        if mm != {q: q for q in range(n)}:
            return False, [f"grover_circuit('{marked}', {it}) should measure qubit q into classical bit q, for every qubit "
                           f"(found {mm or 'no measurements'})."]
        try:
            got = ideal_probabilities(qc)
        except Exception as e:  # noqa: BLE001
            return False, [f"The circuit from grover_circuit('{marked}', {it}) cannot be simulated: {type(e).__name__}: {e}"]
        want = ideal_probabilities(reference_grover(marked, it))
        if np.abs(got - want).max() > 1e-9:
            hint = ""
            flipped = format(int(marked[::-1], 2), f"0{n}b")
            if flipped != marked and np.abs(got - ideal_probabilities(reference_grover(flipped, it))).max() < 1e-9:
                hint = " The circuit marks the reversed string: qubit q is bit marked[n - 1 - q] (qubit 0 is on the right)."
            elif it > 0 and abs(got[int(marked, 2)] - 1 / 2 ** n) < 1e-9:
                hint = (" The marked string is not amplified: check that the circuit runs `iterations` rounds, each with "
                        "the oracle (X gates, mcz, X gates) and the diffuser (H, X, mcz, X, H).")
            elif it > 1 and np.abs(got - ideal_probabilities(reference_grover(marked, it - 1))).max() < 1e-9:
                hint = " It looks like one iteration too few."
            return False, [f"grover_circuit('{marked}', {it}) gives probabilities different from the expected ones "
                           f"(P('{marked}') = {got[int(marked, 2)]:.4f}, expected {want[int(marked, 2)]:.4f}).{hint}"]
        n2, want2 = two_qubit_count(qc), ops_two_qubit_count(reference_grover_ops(marked, it))
        if n2 != want2:
            return False, [f"grover_circuit('{marked}', {it}) gives the right probabilities but has {n2} two-qubit gates; "
                           f"the project's circuit has {want2}. Use the prepared mcz once in each oracle and once in each "
                           "diffuser, and no other two-qubit gates: with noise, every extra gate changes the result."]
    return True, [f"grover_circuit(): passed ({len(GROVER_CASES)} cases, 2 and 3 qubits, 0 to 3 iterations)."]


def _test_counts(my_counts, marked, p):
    if not isinstance(my_counts, dict) or not my_counts:
        return False, ["my_counts is empty. Run the cell of Step 7 that makes it first."]
    total = counts_total(my_counts)
    if total != SHOTS:
        return False, [f"my_counts holds {total} shots; use shots=4000 (Step 7)."]
    keys = list(my_counts)
    if any(len(str(k).replace(" ", "")) != len(marked) for k in keys):
        return False, [f"The keys of my_counts should be {len(marked)}-bit strings such as '{marked}'."]
    k = int(my_counts.get(marked, 0))
    sigma = math.sqrt(p * (1 - p) / SHOTS)
    z = (k / SHOTS - p) / sigma
    if abs(z) > 3:
        return False, [f"In my_counts, {k} of 4000 shots found {marked}: a fraction {k / SHOTS:.4f}, {z:+.1f} standard "
                       f"deviations from the noise model's expected value {p:.4f}. Check that my_counts comes from "
                       f"grover_circuit('{marked}', 2) run with course_noise_model(P2, READOUT) and 4,000 shots. If it does, "
                       "a result this far out happens by chance about 3 times in 1,000: run Step 7 again with another seed."]
    return True, [f"my_counts: {k} of 4000 shots found {marked} ({k / SHOTS:.4f}), within 3 standard deviations "
                  f"({z:+.2f}) of the expected {p:.4f}."]


def personal_value(marked, p2, readout):
    return int(round(10000 * expected_success(format(int(marked), "03b"), 2, float(p2), float(readout))))


def check_l2_m6_grover(grover_circuit, shot_sigma, within_3_sigma, difference_sigma, MARKED, P2, READOUT, my_counts):
    try:
        m = int(MARKED)
        if m != float(MARKED) or not 0 <= m <= 7:
            raise ValueError
    except (TypeError, ValueError):
        return False, ["MARKED is a whole number from 0 to 7. Copy it again from your project quiz."], None
    ok, msg, p2, ro = personal_noise_ok(P2, READOUT)
    if not ok:
        return False, [msg], None
    msgs, passed = [], True
    good, mm = test_statistics(shot_sigma, within_3_sigma, difference_sigma)
    msgs += mm
    passed = passed and good
    good, mm = _test_grover(grover_circuit)
    msgs += mm
    passed = passed and good
    if not passed:
        return False, msgs, None
    marked = format(m, "03b")
    p = expected_success(marked, 2, p2, ro)
    good, mm = _test_counts(my_counts, marked, p)
    msgs += mm
    if not good:
        return False, msgs, None
    return True, msgs, int(round(10000 * p))


passed, messages, value = check_l2_m6_grover(grover_circuit, shot_sigma, within_3_sigma, difference_sigma,
                                             MARKED, P2, READOUT, my_counts)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Step 9: two qubits

With 2 qubits, one iteration finds the marked string with probability exactly 1 in theory, and the circuit has only 2 two-qubit gates (the two `cz`). The cell compares the four possible marked strings under the course noise model.

In [ ]:
for w in ("00", "01", "10", "11"):
    qc = grover_circuit(w, 1)
    p = expected_probabilities(qc, 0.01, 0.02)[w]
    p_gates = expected_probabilities(qc, 0.01, 0.0)[w]
    print(f"marked {w}: expected {p:.4f}   (gate errors only, no readout error: {p_gates:.4f})")

**What to notice.** With only 2 two-qubit gates, readout errors dominate: without them the success would be about 0.98. The marked string 11 does worst and 00 best, because a 1 is misread twice as often as a 0 in this model. Real readout errors are asymmetric in the same way, which is why IBM's software offers **readout error mitigation**: it measures the confusion matrix and corrects the counts for it.

## Limits of this experiment

- **A model, not a computer.** The course model has the same depolarizing error on every gate and qubit, and no errors that depend on time. A real computer has a different error on each qubit pair (Module 4), drifts between calibrations, and has errors that are not random at all (a gate that over-rotates by the same angle every time).
- **No connectivity.** Here any two qubits can share a `cx`. On a real chip, qubits not connected need swaps, which add more two-qubit gates (Module 4); the crossover would come at a lower P2.
- **One marked string, 3 qubits.** With more qubits the best number of iterations grows (about (π/4)√N), the circuit grows with it, and the noise matters even more.
- **Shot noise.** Each measured fraction is uncertain by σ; differences smaller than 3σ_diff are not shown by the data, even when the model says they are there.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level2/QC-L2-M6-hardware-colab.ipynb) runs the project's reference circuit on a real computer and compares the result with the ideal value, with this course's noise model, and with IBM's own noise model of that computer. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days, and one run of 4,000 shots uses a few seconds of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Optional: a short summary of your experiment

Not graded. Writing five short answers is a good way to prepare for the project quiz, and it is how results are reported in a lab notebook or a paper. Double-click this cell to edit it.

1. **Question.** What did you want to find out?
2. **Method.** Which circuit, which noise model (P2, READOUT), how many shots?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.687 ± 0.007, with σ from Step 1).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the Module 6 project quiz for this experiment, Grover's search with noise, and the Module 6 time log.